In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [30]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import StringType, IntegerType

In [ ]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/16 00:24:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/04/16 00:24:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 61659)
Traceback (most recent call last):
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/so

In [17]:
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")
prm_spine = spark.read.parquet("../data/03_primary/prm_spine")
prm_vehicles = spark.read.parquet("../data/03_primary/prm_vehicles")

In [18]:
int_transactions.show()

prm_spine.show()

prm_vehicles.show()

+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+
|      transaction_id|         customer_id| customer_vehicle_id|           branch_id|transaction_dt|      product_family|    product_category|product_sub_category|          product_id|is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehicle|warranty|quantity|net_cost|discount_amount|sales_amount_net|sales_amount|total_profit|sales_amount_net_perc|has_discount|delta_mileage|delta_mileage_perc|
+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+-----------

In [19]:
transactions_data = int_transactions.withColumn(
    "_id",
    f.col("customer_id")
).withColumn(
    "_observ_end_dt",
    f.last_day(f.col("transaction_dt"))
)

In [20]:
base_sales = prm_spine.join(
    transactions_data,
    on=["_id", "_observ_end_dt"],
    how="left"
)

In [21]:
base_sales.show()

+--------------------+--------------+--------------+-----------+-------------------+---------+--------------+--------------+----------------+--------------------+----------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+
|                 _id|_observ_end_dt|transaction_id|customer_id|customer_vehicle_id|branch_id|transaction_dt|product_family|product_category|product_sub_category|product_id|is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehicle|warranty|quantity|net_cost|discount_amount|sales_amount_net|sales_amount|total_profit|sales_amount_net_perc|has_discount|delta_mileage|delta_mileage_perc|
+--------------------+--------------+--------------+-----------+-------------------+---------+--------------+--------------+----------------+--------------------+----------+--------+------+-------

In [79]:
def categorizer(age):
    if age is None:
        return "None"
    elif age < 3:
        return "0_2"
    elif age < 6:
        return "3_5"
    elif age < 10:
        return "6_9"
    else:
        return "10_plus"

In [89]:
bucket_udf = f.udf(categorizer, StringType())

ftr_vehicles = base_sales.join(
    prm_vehicles,
    on=["customer_id", "customer_vehicle_id"],
    how="left"
)

ftr_vehicles = ftr_vehicles.withColumn(
    "car_age",
    f.year(f.col("transaction_dt")).cast("int") - f.col("model_year").cast("int")
).withColumn(
    "car_age_group",
    bucket_udf(f.col("car_age"))
).withColumn(
    "car_group",
    f.concat(f.lower(f.col("vehicle_brand_level")), f.lit("__"), f.col("car_age_group"))
)

In [106]:
car_price_levels = [f"car_brand__{col}" for col in ["very_low", "low", "medium", "high", "very_high"]]
car_age_levels = [f"car_age__{col}" for col in [None, "0_2", "3_5", "6_9", "10_plus"]]
car_price_age_levels = [
    f"{price}__{age}"
    for price in car_price_levels
    for age in car_age_levels
]
car_age_levels

['car_age__None',
 'car_age__0_2',
 'car_age__3_5',
 'car_age__6_9',
 'car_age__10_plus']

In [95]:
len(car_price_age_levels)

25

In [110]:
ftr_vehicles_price_pivot = ftr_vehicles.dropna(
    subset=["_id", "customer_vehicle_id", "transaction_id"]
).withColumn(
    "vehicle_brand_level",
    f.concat(f.lit("car_brand__"), f.lower(f.col("vehicle_brand_level")))
).groupBy(
    "_id", "customer_vehicle_id", "transaction_id"
).pivot(
    "vehicle_brand_level", values=car_price_levels
).agg(
    f.countDistinct("customer_id").alias("count")
).fillna(0)

ftr_vehicles_age_pivot = ftr_vehicles.dropna(
    subset=["_id", "customer_vehicle_id", "transaction_id"]
).withColumn(
    "car_age_group",
    f.concat(f.lit("car_age__"), f.col("car_age_group"))
).groupBy(
    "_id", "customer_vehicle_id", "transaction_id"
).pivot(
    "car_age_group", values=car_age_levels
).agg(
    f.countDistinct("customer_id").alias("count")
).fillna(0)

ftr_vehicles_price_pivot.show()
ftr_vehicles_age_pivot.show()

+--------------------+--------------------+--------------------+-------------------+--------------+-----------------+---------------+--------------------+
|                 _id| customer_vehicle_id|      transaction_id|car_brand__very_low|car_brand__low|car_brand__medium|car_brand__high|car_brand__very_high|
+--------------------+--------------------+--------------------+-------------------+--------------+-----------------+---------------+--------------------+
|7591E770-A218-427...|F64B83C2-2B22-4F6...|1DE4485D-5F4F-48F...|                  0|             0|                0|              0|                   0|
|A0EBF44D-5041-4AC...|EB3AB56F-A315-40C...|366A1EC0-A7BF-40D...|                  0|             0|                0|              0|                   0|
|560534E5-B132-4FC...|EF8E6596-6E93-4E0...|56327B9F-C65B-412...|                  0|             0|                1|              0|                   0|
|6F7B5D13-A4E9-433...|7895DDF8-86FE-4A0...|87EFA09D-9907-4B0...|      

+--------------------+--------------------+--------------------+-------------+------------+------------+------------+----------------+
|                 _id| customer_vehicle_id|      transaction_id|car_age__None|car_age__0_2|car_age__3_5|car_age__6_9|car_age__10_plus|
+--------------------+--------------------+--------------------+-------------+------------+------------+------------+----------------+
|E3D71FD7-8D7E-4A5...|846E34DA-036C-437...|B05D6CE9-78F3-4D0...|            0|           0|           1|           0|               0|
|1E9A6177-BB81-4CA...|6BBFC656-251C-4BA...|5F5AF145-772E-4CB...|            0|           0|           0|           0|               1|
|00191B83-196D-44B...|2418E360-E920-470...|3F13EFB0-121B-43C...|            0|           0|           0|           0|               1|
|9196DE26-E619-4BC...|7EDB3AB1-5495-450...|022C4E57-EE69-48D...|            0|           1|           0|           0|               0|
|F79F138D-7C07-426...|D371C1A7-8E97-40F...|5F691C58-AEC

25/04/16 03:49:27 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 891744 ms exceeds timeout 120000 ms
25/04/16 03:49:27 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/16 03:49:33 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [101]:
ftr_vehicles.dropna(
    subset=["_id", "customer_vehicle_id", "transaction_id"]
).withColumn(
    "quarter",
    f.to_date(f.date_trunc("quarter", f.col("transaction_dt")))
).show()

+--------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------+--------------------+-------------------+--------------------+--------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+-------+------------+-------------------+----------+-----------------+------------+-----------------+-------+-------------+--------------+----------+
|         customer_id| customer_vehicle_id|                 _id|_observ_end_dt|      transaction_id|           branch_id|transaction_dt|      product_family|   product_category|product_sub_category|          product_id|is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehicle|warranty|quantity|net_cost|discount_amount|sales_amount_net|sales_amount|total_profit|sales_amount_net_perc|ha

In [58]:
ftr_vehicles.filter(
    # f.col("transaction_id").isNotNull()
    f.col("_id") == "000022BE-0CD3-4413-8FD5-4A58BA75BA05"
).orderBy(
    "_id", "_observ_end_dt", "customer_vehicle_id",
).show(truncate=False)

+------------------------------------+------------------------------------+--------------+------------------------------------+------------------------------------+------------------------------------+--------------+-------------------------+------------------+--------------------------------------------+--------------------------------------------------------------------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+------------------------------------+------+-----------+-------------------+----------+-----------------+------------+-----------------+-------+-------------+-----------+
|customer_vehicle_id                 |_id                                 |_observ_end_dt|transaction_id                      |customer_id                         |branch_id                           |transaction_d

In [ ]:
ftr_vehicles.select(
    "_id",
    "_observ_end_dt",
    "customer_id",
    "customer_vehicle_id",
    ""
)

In [13]:
sales_branch_view.show(5, False)

+------------------------------------+--------------+------------------------------------+--------------+
|_id                                 |_observ_end_dt|branch_id                           |trx_per_branch|
+------------------------------------+--------------+------------------------------------+--------------+
|4CDAF1D5-512D-4EAE-931B-53F9DF678EBF|2024-01-31    |70919EBC-05BE-4CCF-8DF6-C6BCAC47BBA4|1             |
|7E9833DC-A403-4B2B-B267-0B1704B73A48|2024-06-30    |B1323649-5458-41A4-B03D-C584CE74AF3E|1             |
|076844A0-0120-4467-B805-272235724D56|2024-10-31    |FE6D471C-A328-49F1-A769-A2ADD53D24FC|1             |
|D3548A7A-8AE4-42A4-BBCF-1185F17B292A|2024-02-29    |1FCCE949-B97D-440C-BE1B-FFB03D0267CE|1             |
|CBA38EBA-F74C-45AE-B6A3-59D1ABA0966C|2024-05-31    |CEEFC64C-F2BA-48C9-AF65-1782F56B7D3E|1             |
+------------------------------------+--------------+------------------------------------+--------------+
only showing top 5 rows



In [18]:
sales_branch_geolocation = sales_branch_view.join(
    prm_geolocation.dropDuplicates(),
    on="branch_id",
    how="left"
)
sales_branch_geolocation.show(truncate=False)

+------------------------------------+------------------------------------+--------------+--------------+------------------+------------------+-----------+-----------+---------+--------+-------------------------------+-----------------------------------------------+-----------------------------------+--------------------------------+-------------------------------+----------------------------------+------------------------------+---------------------------------+------------------------------------+-------------------------+------------------------------+----------------------------------------------+----------------------------------+-------------------------------+------------------------------+---------------------------------+-----------------------------+--------------------------------+-----------------------------------+------------------------+------------------------------+----------------------------------------------+----------------------------------+-----------------------

In [19]:
from pyspark.sql import Window

In [22]:
w = Window.partitionBy('_id', '_observ_end_dt').orderBy(f.col('trx_per_branch').desc())

In [42]:
select_list = [
    f.col(col).alias(f"most_trx_branch_{col}")
    for col in geolocations_cols_to_keep
    if col not in ["_id", "_observ_end_dt",]
]

In [43]:
ftr_branch_most_trasactions_geolocation = sales_branch_geolocation.withColumn(
    'row_number',
    f.row_number().over(w)
).filter(
    f.col("trx_per_branch") > 0
).orderBy(
    "_id", "_observ_end_dt", "row_number"
).filter(
    f.col('row_number') == 1
).select("_id", "_observ_end_dt", *select_list)

ftr_branch_most_trasactions_geolocation.show(10)

+--------------------+--------------+-------------------------+-----------------------------------------------+---------------------------------------------------------------+---------------------------------------------------+------------------------------------------------+-----------------------------------------------+--------------------------------------------------+----------------------------------------------+-------------------------------------------------+----------------------------------------------------+-----------------------------------------+----------------------------------------------+--------------------------------------------------------------+--------------------------------------------------+-----------------------------------------------+----------------------------------------------+-------------------------------------------------+---------------------------------------------+------------------------------------------------+--------------------------------

In [32]:
geolocations_cols_to_drop = ["trx_per_branch", "longitude", "latitude","branch_code","branch_type","is_active","city"]
geolocations_cols_to_keep = [col for col in sales_branch_geolocation.columns if col not in geolocations_cols_to_drop]

In [35]:
agg_list = [f.avg(f.col(col)).alias(f"avg_{col}") for col in geolocations_cols_to_keep if col not in ["_id", "_observ_end_dt", "branch_id"]]

In [38]:

ftr_avg_branch_geolocation = sales_branch_geolocation.filter(
    f.col("trx_per_branch") > 0
).drop(
    *geolocations_cols_to_drop
).groupby(
    "_id", "_observ_end_dt"
).agg(*agg_list)

ftr_avg_branch_geolocation.show(truncate=False)

+------------------------------------+--------------+-----------------------------------+---------------------------------------------------+---------------------------------------+------------------------------------+-----------------------------------+--------------------------------------+----------------------------------+-------------------------------------+----------------------------------------+-----------------------------+----------------------------------+--------------------------------------------------+--------------------------------------+-----------------------------------+----------------------------------+-------------------------------------+---------------------------------+------------------------------------+---------------------------------------+----------------------------+----------------------------------+--------------------------------------------------+--------------------------------------+-----------------------------------+----------------------------

In [40]:
transactions_data.filter(
    f.col("_id") == "F23BA7E3-9FB6-4A93-8941-E92B8358F21B"
).filter(
    f.col("_observ_end_dt") == "2024-09-30"
).orderBy("_id", "_observ_end_dt").show(50, truncate=False)

+------------------------------------+------------------------------------+------------------------------------+------------------------------------+--------------+--------------------------------------+-----------------+--------------------+---------------------------------------------------------------------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+------------------------------------+--------------+
|transaction_id                      |customer_id                         |customer_vehicle_id                 |branch_id                           |transaction_dt|product_family                        |product_category |product_sub_category|product_id                                                                       |is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehic

In [41]:
prm_geolocation.filter(
    f.col("branch_id") == "EE5B9A50-31AF-4854-B259-6C8925347D21"
).show(50, truncate=False)

+------------------------------------+------------------+-----------------+-----------+-----------+---------+------+-------------------------------+-----------------------------------------------+-----------------------------------+--------------------------------+-------------------------------+----------------------------------+------------------------------+---------------------------------+------------------------------------+-------------------------+------------------------------+----------------------------------------------+----------------------------------+-------------------------------+------------------------------+---------------------------------+-----------------------------+--------------------------------+-----------------------------------+------------------------+------------------------------+----------------------------------------------+----------------------------------+-------------------------------+------------------------------+------------------------------

In [46]:
out = base_sales.select("_id", "_observ_end_dt").join(
    ftr_branch_most_trasactions_geolocation,
    on=["_id", "_observ_end_dt"],
    how="left"
).join(
    ftr_avg_branch_geolocation,
    on=["_id", "_observ_end_dt"],
    how="left"
).orderBy("_id", "_observ_end_dt")

In [47]:
out.show()

+--------------------+--------------+-------------------------+-----------------------------------------------+---------------------------------------------------------------+---------------------------------------------------+------------------------------------------------+-----------------------------------------------+--------------------------------------------------+----------------------------------------------+-------------------------------------------------+----------------------------------------------------+-----------------------------------------+----------------------------------------------+--------------------------------------------------------------+--------------------------------------------------+-----------------------------------------------+----------------------------------------------+-------------------------------------------------+---------------------------------------------+------------------------------------------------+--------------------------------